In [3]:
!pip install opencv-python

  Using cached opencv_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.10.0.84-cp37-abi3-win_amd64.whl (38.8 MB)


In [2]:
!pip install --upgrade tensorflow

  Using cached absl_py-2.1.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-24.12.23-py2.py3-none-any.whl.metadata (876 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-2.5.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached tensorboard-2.18.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached namex-0.0.8-py3-none-any.whl.metadata (246 bytes)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
   ---------------------------------------- 0.0/390.3 MB ? eta -:--:--
   - -------------------------------------- 15.5/390.3 MB 88.4 MB/s eta 0:00:05
   --- ------------------------------------ 37.2/390.3 MB 94.5 MB/s eta 0:00:04


In [6]:
# train_model.py

import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# Ustawienia
HR_IMAGE_FOLDER = "C:/Users/adida/OneDrive/Pulpit/archive/DIV2K_train_HR/DIV2K_train_HR"   # folder z obrazami wysokiej rozdzielczości (HR)
MODEL_SAVE_PATH = "C:/Users/adida/OneDrive/Pulpit/archive/Test-Output/super_res_model.h5"
BATCH_SIZE = 4
EPOCHS = 10          # Dla przykładu ustawiamy małą liczbę – realnie dostosuj do potrzeb
LR_SCALE = 0.5      # współczynnik skalowania w dół do LR (możesz zmienić np. na 0.25)

def load_image_paths(folder):
    """Zwraca listę ścieżek do plików w danym folderze."""
    all_files = []
    for f in os.listdir(folder):
        full_path = os.path.join(folder, f)
        if os.path.isfile(full_path) and f.lower().endswith(('.png', '.jpg', '.jpeg')):
            all_files.append(full_path)
    return all_files

def data_generator(hr_folder, batch_size=4):
    """
    Generator do trenowania:
    - Wczytuje obrazy HR
    - Skaluje je w dół do LR
    - Zwraca pary (LR, HR) w mini-batchach
    """
    image_paths = load_image_paths(hr_folder)
    while True:
        np.random.shuffle(image_paths)
        batch_lr = []
        batch_hr = []
        for path in image_paths:
            hr_img = cv2.imread(path)
            hr_img = cv2.cvtColor(hr_img, cv2.COLOR_BGR2RGB)
            hr_img = hr_img.astype(np.float32) / 255.0

            # Skalowanie w dół
            h, w, _ = hr_img.shape
            lr_img = cv2.resize(hr_img, (int(w*LR_SCALE), int(h*LR_SCALE)), interpolation=cv2.INTER_CUBIC)

            # Dostosowujemy obrazy do stałego rozmiaru (przykładowo 128x128 na wyjściu)
            hr_img_resized = cv2.resize(hr_img, (1920, 1080), interpolation=cv2.INTER_CUBIC)
            lr_img_resized = cv2.resize(lr_img, (960, 540), interpolation=cv2.INTER_CUBIC)

            batch_lr.append(lr_img_resized)
            batch_hr.append(hr_img_resized)

            if len(batch_lr) == batch_size:
                yield np.array(batch_lr), np.array(batch_hr)
                batch_lr = []
                batch_hr = []

def build_model():
    """
    Prosty model CNN do super-rezolucji:
    - Wejście: (64, 64, 3)
    - Kilka warstw konwolucyjnych
    - UpSampling do 128x128
    """
    inputs = layers.Input(shape=(540, 960, 3))

    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(inputs)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.UpSampling2D(size=(2,2), interpolation='nearest')(x)  # skala x2
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    outputs = layers.Conv2D(3, (3,3), padding='same', activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def main():
    # Budujemy model
    model = build_model()
    model.summary()

    # Generator danych
    train_gen = data_generator(HR_IMAGE_FOLDER, BATCH_SIZE)

    # Obliczenie steps_per_epoch
    num_images = len(load_image_paths(HR_IMAGE_FOLDER))
    steps_per_epoch = num_images // BATCH_SIZE if num_images > 0 else 1

    # Trening
    model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        epochs=EPOCHS
    )

    # Zapisujemy wytrenowany model
    model.save(MODEL_SAVE_PATH)
    print(f"Model zapisany do: {MODEL_SAVE_PATH}")

if __name__ == "__main__":
    main()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 540, 960, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 540, 960, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 540, 960, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_3 (UpSampling2D)  │ (None, 1080, 1920, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 1080, 1920, 64) │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 1080, 1920, 3)  │         1,731 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,379 (302.26 KB)

 Trainable params: 77,379 (302.26 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
 16/200 ━━━━━━━━━━━━━━━━━━━━ 39:37 13s/step - loss: 0.0821

KeyboardInterrupt: 

In [9]:
model = tf.keras.models.load_model("C:/Users/adida/OneDrive/Pulpit/archive/Test_Output/super_res_model.h5", compile=False)
model.compile(optimizer='adam', loss='mse', metrics=['accuracy'])

In [10]:
import cv2
import numpy as np
import tensorflow as tf

MODEL_PATH = "C:/Users/adida/OneDrive/Pulpit/archive/Test_Output/super_res_model.h5"

def enhance_image(input_path, output_path):
    # Wczytanie wytrenowanego modelu
    model = tf.keras.models.load_model(MODEL_PATH)

#Wczytujemy obraz do poprawy
    img = cv2.imread(input_path)
    if img is None:
        print(f"[ERROR] Nie można wczytać obrazu z: {input_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0

    # Zmniejszamy do (64, 64)
    img_resized = cv2.resize(img, (960, 540), interpolation=cv2.INTER_CUBIC)
    img_resized = np.expand_dims(img_resized, axis=0)  # (1, 64, 64, 3)

    # Przepuszczamy przez sieć
    pred = model.predict(img_resized)[0]  # (128, 128, 3)

    # Skalujemy z powrotem do [0, 255]
    pred = np.clip(pred * 255.0, 0, 255).astype(np.uint8)

    # Zapis do pliku
    pred_bgr = cv2.cvtColor(pred, cv2.COLOR_RGB2BGR)
    cv2.imwrite(output_path, pred_bgr)
    print(f"[INFO] Zapisano poprawiony obraz: {output_path}")

def main():
    # Tu na sztywno ustalasz ścieżki
    input_path = r"C:/Users/adida/OneDrive/Pulpit/archive/DIV2K_train_HR/witcher1.jpg"
    output_path = r"C:/Users/adida/OneDrive/Pulpit/archive/DIV2K_train_HR/5.jpg"

    enhance_image(input_path, output_path)

if __name__ == "__main__":
    main()

1/1 ━━━━━━━━━━━━━━━━━━━━ 452s 452s/step
[INFO] Zapisano poprawiony obraz: C:/Users/adida/OneDrive/Pulpit/archive/DIV2K_train_HR/5.jpg
